In [1]:
!pip install pygame

In [4]:
import numpy
import pygame
import math
import time
from utils import *
from random import *
import sys


# The grass background auemntado 2.5 vezes mais
GRASS = scale_image(pygame.image.load("imgs/grass.jpg"), 2.5)
# The track image diminuida em 90%
TRACK = scale_image(pygame.image.load("imgs/track.png"), 0.9)

TRACK_MASK = scale_image(pygame.image.load("imgs/track-mask.png"), 0.9)

# The track border image for collision detection diminuída em 90%
TRACK_BORDER = scale_image(pygame.image.load("imgs/track-border.png"), 0.9)
TRACK_BORDER_MASK=pygame.mask.from_surface(TRACK_BORDER)

# The finish image intacta
FINISH = pygame.image.load("imgs/finish.png")
FINISH_MASK=pygame.mask.from_surface(FINISH)

# import cars images diminuídas em 55%
RED_CAR = scale_image(pygame.image.load("imgs/red-car.png"), 0.55)
GREEN_CAR = scale_image(pygame.image.load("imgs/green-car.png"), 0.55)
car_width,car_height=GREEN_CAR.get_size()
### half car-width and half car-height
HALF_WIDTH=car_width/2
HALF_HEIGHT=car_height/2
CAR_SIZE=HALF_WIDTH,HALF_HEIGHT

# Get the size of the track image
WIDTH, HEIGHT = TRACK.get_width(), TRACK.get_height()

# the window has the size of the track image
WIN = pygame.display.set_mode((WIDTH, HEIGHT))

# the name of the window
pygame.display.set_caption("Racing Game!")

pygame.font.init()
MAIN_FONT = pygame.font.SysFont("comicsans", 44)

FPS=60

RED = (255, 0, 0, 255)
WHITE = (255, 255, 255, 255)
YELLOW = (255, 255, 0, 255)

FINISH_POSITION=(130,250)
images=[(GRASS,(0,0)),(TRACK,(0,0)),(FINISH,FINISH_POSITION),(TRACK_BORDER,(0,0))]


def draw(win,images,player_car,computer_car,game_info):
    for img,pos in images:
        win.blit(img,pos)
    level_text=MAIN_FONT.render(f'Level {game_info.level}',1,(255,255,255))
    win.blit(level_text,(10,HEIGHT-level_text.get_height()-90))
    
    time_text=MAIN_FONT.render(f'Time {game_info.get_level_time()}',1,(255,255,255))
    win.blit(time_text,(10,HEIGHT-time_text.get_height()-50))
    
    velocity_text=MAIN_FONT.render(f'Vel {round(computer_car.vel,1)} px/s',1,(255,255,255))
    win.blit(velocity_text,(10,HEIGHT-velocity_text.get_height()-10))
    
    player_car.draw(win)
    computer_car.draw(win)
    pygame.display.update()
        
class AbstractCar:
    
    def __init__(self, max_vel, rotation_vel):
        self.img = self.IMG
        self.max_vel = max_vel
        self.vel = 0
        self.rotation_vel = rotation_vel
        self.angle = 0
        self.x,self.y=self.START_POS
        self.acceleration=1

    def rotate(self, left=False, right=False):
        if left and right:
            pass
        elif left:
            self.angle += self.rotation_vel + choice(range(-1,1))
        elif right:
            self.angle -= self.rotation_vel +  choice(range(-1,1))
        self.angle = int(self.angle) % 360
        
    def move_forward(self):
        self.vel = min(self.vel + self.acceleration, self.max_vel)
        self.move()
        
    def move_backwards(self):
        self.vel = max(self.vel - self.acceleration, -self.max_vel//2)
        self.move()
    
    
    # primeiro calculamos o  ângulo em radianos
    # Calculamos o deslocamento em x e y através da trignometra
    # actualizamos x e y mas subtraindo devido aos pontos cardeais do pygame e da corrida de carros
    def move(self):
        radians = math.radians(self.angle)
        vertical = math.cos(radians) * self.vel
        horizontal = math.sin(radians) * self.vel

        self.y -= vertical
        self.x -= horizontal
        
    
    def collide(self, mask, x=0, y=0):
        # 1. Roda a imagem atual com o ângulo atual do carro
        rotated_image = pygame.transform.rotate(self.img, self.angle)

        # 2. Mantém o centro da imagem rodada no mesmo sítio do centro da imagem original
        new_rect = rotated_image.get_rect(center=self.img.get_rect(topleft=(self.x, self.y)).center)

        # 3. Cria a máscara e calcula o offset com as novas coordenadas
        car_mask = pygame.mask.from_surface(rotated_image)
        offset = (int(new_rect.x - x), int(new_rect.y - y))

        poi = mask.overlap(car_mask, offset)
        return poi
        
    def reset(self):
        self.x,self.y=self.START_POS
        self.angle=0
        self.vel=0

    # x,y will be the center of the car
    #
    def draw(self, win):
        blit_rotate_center(win, self.img, (self.x, self.y), self.angle)
        #for b in self.bullets:
        #    b.draw(win)
        x, y = int(self.x + CAR_SIZE[0]), int(self.y + CAR_SIZE[1])
        dx, dy = CAR_SIZE[0], CAR_SIZE[1]
        alfa = (self.angle) * math.pi / 180
        mrot = numpy.array([[math.cos(alfa), -math.sin(alfa)], [math.sin(alfa), math.cos(alfa)]])
        pts = numpy.array([[-dx, -dy], [dx, -dy], [-dx, dy], [dx, dy]])
        npts = numpy.dot(pts, mrot)
        for i in npts:
            if 0<=i[0]+x<WIDTH and 0<=i[1]+y<HEIGHT: 
                pygame.draw.circle(win, TRACK_MASK.get_at((int(i[0] + x), int(i[1] + y))), \
                                                          (int(i[0] + x), int(i[1] + y)),2, 2)
        


class PlayerCar(AbstractCar):
    IMG = RED_CAR
    START_POS = (180, 200)
    
    def reduce_speed(self):
        self.vel = max(self.vel - self.acceleration / 2, 0)
        self.move()
        
    def bounce(self):
        self.vel=-self.vel
        self.move()
    
PATH=[(161, 148), (147, 82), (68, 93), (61, 174), (64, 251), (59, 464), (293, 711), (399, 712), (407, 537), (491, 474), (591, 529), (618, 707), (731, 709), (739, 383), (395, 334), (438, 250), (706, 251), (738, 96), (287, 93), (273, 395), (179, 399), (173, 258)]

        
class ComputerCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)
    
    def __init__(self, max_vel, rotation_vel, path=[]):
        super().__init__(max_vel, rotation_vel)
        self.path=path
        self.current_point=0
        self.vel=self.max_vel
             
    def calculate_angle(self):
        target_x, target_y = self.path[self.current_point]
        x_diff = target_x - self.x
        y_diff = target_y - self.y

        if y_diff == 0:
            desired_radian_angle = (1 if x_diff<0 else -1)*math.pi / 2
        else:
            desired_radian_angle = math.atan(x_diff / y_diff)

        if target_y > self.y:
            desired_radian_angle += math.pi

        difference_in_angle = self.angle - math.degrees(desired_radian_angle)
        if difference_in_angle >= 180:
            difference_in_angle -= 360
            
        elif difference_in_angle <= -180:
            difference_in_angle += 360

        if difference_in_angle > 0:
            self.angle -= min(self.rotation_vel, abs(difference_in_angle))
        else:
            self.angle += min(self.rotation_vel, abs(difference_in_angle))

    def update_path_point(self):
        target = self.path[self.current_point]
        rect = pygame.Rect(
            self.x, self.y, self.img.get_width(), self.img.get_height())
        if rect.collidepoint(*target):
            self.current_point += 1
            
    def move(self):
        if self.current_point >= len(self.path):
            return

        self.calculate_angle()
        self.update_path_point()
        super().move()
            
    def next_level(self,level):
        self.reset()
        self.vel=self.max_vel+(level-1)*0.02
        self.current_point=0
        
        
    def draw_points(self,win):
        for point in self.path:
            pygame.draw.circle(win,(255,0,0),point,5)
        
    def draw(self,win):
        super().draw(win)
        # self.draw_points(win)
        
        

def move_player(player_car):
    keys=pygame.key.get_pressed()
    moved=False
    if keys[pygame.K_a]:
        player_car.rotate(left=True)
    elif keys[pygame.K_d]:
        player_car.rotate(right=True)
    elif keys[pygame.K_w]:
        moved=True
        player_car.move_forward()
    elif keys[pygame.K_s]:
        moved=True
        player_car.move_backwards()
    if not moved:
        player_car.reduce_speed()


def handle_collision(player_car, computer_car,game_info):
    if player_car.collide(TRACK_BORDER_MASK) != None:
        player_car.bounce()

    computer_finish_poi_collide = computer_car.collide(
        FINISH_MASK, *FINISH_POSITION)
    if computer_finish_poi_collide != None:
        blit_text_center(WIN,MAIN_FONT,"YOU LOST!")
        pygame.display.update()
        pygame.time.wait(5000)
        game_info.reset()
        player_car.reset()
        computer_car.next_level(1)
        return True

    player_finish_poi_collide = player_car.collide(
        FINISH_MASK, *FINISH_POSITION)
    if player_finish_poi_collide != None:
        if player_finish_poi_collide[1] == 0:
            player_car.bounce()
        else:
            player_car.reset()
            game_info.next_level()
            computer_car.next_level(game_info.level)
            return True
    return False


class GameInfo:
    LEVELS = 10

    def __init__(self, level=1):
        self.level = level
        self.started = False
        self.level_start_time = 0

    def next_level(self):
        self.level += 1
        self.started = False

    def reset(self):
        self.level = 1
        self.started = False
        self.level_start_time = 0

    def game_finished(self):
        return self.level > self.LEVELS

    def start_level(self):
        self.started = True
        self.level_start_time = time.time()

    def get_level_time(self):
        if not self.started:
            return 0
        return round(time.time() - self.level_start_time)



In [5]:
import neat
import os

In [6]:
class NeatCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)

    def __init__(self, max_vel, rotation_vel, start_angle=0, track_reversed=False):
        super().__init__(max_vel, rotation_vel)
        self.angle = start_angle
        self.x, self.y = (150, 200)
        self.alive = True
        self.distance = 0
        self.current_waypoint = 0
        self.finished = False
        self.finish_time = 0

    def get_waypoint_data(self, waypoints):
        if len(waypoints) == 0:
            return [0, 0]
        
        target_x, target_y = waypoints[self.current_waypoint]
        
        # Input 1: Distância normalizada
        dist = math.hypot(target_x - self.x, target_y - self.y)
        dist_normalizada = dist / math.hypot(WIDTH, HEIGHT)
        
        # Input 2: Ângulo relativo ao próximo waypoint
        x_diff = target_x - self.x
        y_diff = target_y - self.y
        if y_diff == 0:
            desired_angle = (1 if x_diff < 0 else -1) * 90
        else:
            desired_angle = math.degrees(math.atan(x_diff / y_diff))
        if target_y > self.y:
            desired_angle += 180
        angle_diff = (self.angle - desired_angle) % 360
        if angle_diff > 180:
            angle_diff -= 360
        angle_normalizado = -angle_diff / 180

        return [dist_normalizada, angle_normalizado]

    def update_waypoint(self, waypoints):
        target_x, target_y = waypoints[self.current_waypoint]
        dist = math.hypot(target_x - self.x, target_y - self.y)
        if dist < 25:
            self.current_waypoint += 1
            if self.current_waypoint >= len(waypoints):
                self.current_waypoint = 0

    def apply_nn_actions(self, throttle, steering):
        noise_throttle = uniform(-0.05, 0.05)
        noise_steering = uniform(-0.05, 0.05)
        throttle = max(-1.0, min(1.0, throttle + noise_throttle))
        steering = max(-1.0, min(1.0, steering + noise_steering))
        if throttle > 0:
            self.vel = min(self.vel + self.acceleration * throttle, self.max_vel)
        else:
            self.vel = max(self.vel + (self.acceleration * 2) * throttle, 0)
        self.angle += self.rotation_vel * steering
        self.angle = int(self.angle) % 360
        self.move()
        self.distance += self.vel

    def check_collision(self, frame_count):
        if self.collide(TRACK_BORDER_MASK) != None:
            self.alive = False
            self.vel = 0
        if frame_count > 180:
            finish_poi_collide = self.collide(FINISH_MASK, *FINISH_POSITION)
            if finish_poi_collide != None:
                if finish_poi_collide[1] == 0:
                    self.alive = False
                    self.vel = 0
                else:
                    self.finished = True
                    self.vel = 0
                    self.finish_time = frame_count / FPS

    def draw(self, win):
        super().draw(win)

In [7]:
pygame.init()
global WIN
WIN = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Marca os Waypoints!")

waypoints = []
run = True
clock = pygame.time.Clock()

print("Clica na pista para adicionar waypoints. Prime ENTER para terminar.")

while run:
    clock.tick(FPS)
    
    WIN.blit(GRASS, (0, 0))
    WIN.blit(TRACK, (0, 0))

    for i, point in enumerate(waypoints):
        pygame.draw.circle(WIN, (255, 0, 0), point, 6)
        if i > 0:
            pygame.draw.line(WIN, (255, 0, 0), waypoints[i-1], point, 2)

    pygame.display.update()

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            run = False
            pygame.quit()
            sys.exit()

        if event.type == pygame.MOUSEBUTTONDOWN:
            pos = pygame.mouse.get_pos()
            waypoints.append(pos)
            print(f"Waypoint {len(waypoints)}: {pos}")

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_RETURN:
                run = False

print(f"\nWaypoints guardados: {waypoints}")

Clica na pista para adicionar waypoints. Prime ENTER para terminar.
Waypoint 1: (173, 171)
Waypoint 2: (160, 107)
Waypoint 3: (115, 80)
Waypoint 4: (41, 114)
Waypoint 5: (49, 188)
Waypoint 6: (60, 271)
Waypoint 7: (65, 370)
Waypoint 8: (73, 459)
Waypoint 9: (131, 551)
Waypoint 10: (233, 635)
Waypoint 11: (324, 707)
Waypoint 12: (394, 731)
Waypoint 13: (438, 677)
Waypoint 14: (453, 591)
Waypoint 15: (444, 515)
Waypoint 16: (510, 489)
Waypoint 17: (617, 578)
Waypoint 18: (632, 711)
Waypoint 19: (724, 685)
Waypoint 20: (750, 525)
Waypoint 21: (730, 371)
Waypoint 22: (548, 379)
Waypoint 23: (416, 350)
Waypoint 24: (433, 249)
Waypoint 25: (618, 244)
Waypoint 26: (728, 219)
Waypoint 27: (706, 95)
Waypoint 28: (474, 61)
Waypoint 29: (339, 59)
Waypoint 30: (280, 173)
Waypoint 31: (263, 318)
Waypoint 32: (183, 389)
Waypoint 33: (161, 237)

Waypoints guardados: [(173, 171), (160, 107), (115, 80), (41, 114), (49, 188), (60, 271), (65, 370), (73, 459), (131, 551), (233, 635), (324, 707), (394, 731

In [8]:
def eval_genomes_waypoints(genomes, config):
    INVERTER_PISTA = False
    ANGULO_INICIAL = 180 if INVERTER_PISTA else 0

    nets = []
    cars = []
    ge = []

    for genome_id, genome in genomes:
        genome.fitness = 0
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        nets.append(net)
        cars.append(NeatCar(max_vel=4, rotation_vel=5.5, start_angle=ANGULO_INICIAL, track_reversed=INVERTER_PISTA))
        ge.append(genome)

    clock = pygame.time.Clock()
    frame_count = 0

    selected_waypoint = None
    while len(cars) > 0:
        clock.tick(FPS)
        frame_count += 1

        if frame_count > 20000:
            break

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            
            if event.type == pygame.MOUSEBUTTONDOWN:
                mouse_x, mouse_y = pygame.mouse.get_pos()
                for j, point in enumerate(waypoints):
                    if math.hypot(mouse_x - point[0], mouse_y - point[1]) < 15:
                        selected_waypoint = j
                        break
            
            if event.type == pygame.MOUSEBUTTONUP:
                selected_waypoint = None
            
            if event.type == pygame.MOUSEMOTION and selected_waypoint is not None:
                waypoints[selected_waypoint] = pygame.mouse.get_pos()

        WIN.blit(GRASS, (0, 0))
        WIN.blit(TRACK, (0, 0))
        WIN.blit(FINISH, FINISH_POSITION)

        for point in waypoints:
            pygame.draw.circle(WIN, (255, 0, 0), point, 5)

        for i in reversed(range(len(cars))):
            car = cars[i]

            old_waypoint = car.current_waypoint
            car.update_waypoint(waypoints)
            inputs = car.get_waypoint_data(waypoints)
            output = nets[i].activate(inputs)

            throttle = output[0]
            steering = output[1]

            if abs(steering) < 0.2:
                steering = 0

            car.apply_nn_actions(throttle, steering)
            car.check_collision(frame_count)

            if frame_count == 180:
                dist_start = math.hypot(car.x - 150, car.y - 200)
                if dist_start < 100:
                    car.alive = False

            if car.finished:
                ge[i].fitness += 10000
                print(f"\n[!] SUCESSO! O carro encontrou a meta em {car.finish_time:.2f} segundos!")
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)
                break

            elif not car.alive or (car.vel <= 0 and frame_count > 60):
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)

            else:
                if car.current_waypoint != old_waypoint:
                    ge[i].fitness += 200

                car.draw(WIN)
                wp_font = pygame.font.SysFont("comicsans", 20)
                wp_txt = wp_font.render(str(car.current_waypoint), 1, (255, 255, 0))
                WIN.blit(wp_txt, (int(car.x), int(car.y)))

        pygame.display.update()

In [ ]:
def run_neat_waypoints(config_path):
    pygame.init()
    global WIN
    WIN = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Racing Game - Waypoints!")

    config = neat.config.Config(neat.DefaultGenome, neat.DefaultReproduction,
                                neat.DefaultSpeciesSet, neat.DefaultStagnation,
                                config_path)

    p = neat.Population(config)

    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    import time
    start_time = time.time()
    winner = p.run(eval_genomes_waypoints, 25)
    elapsed_time = time.time() - start_time

    pygame.quit()

    try:
        import pickle
        with open('winner_waypoints.pkl', 'wb') as f:
            pickle.dump(winner, f)

        info = {
            'melhor_fitness': winner.fitness,
            'melhor_geracao': stats.best_genome_generation(),
            'tempo_total': elapsed_time,
            'num_waypoints': len(waypoints),
            'config': config_path
        }
        with open('stats_waypoints.pkl', 'wb') as f:
            pickle.dump(info, f)

        print(f"\nMelhor fitness: {winner.fitness}")
        print(f"Melhor geração: {len(stats.most_fit_genomes)}")
        print(f"Tempo total: {elapsed_time:.1f} segundos")
        print(f"Número de waypoints: {len(waypoints)}")
    except Exception as e:
        print(f"Erro ao guardar: {e}")

    print('\nMelhor genoma encontrado:\n{!s}'.format(winner))

run_neat_waypoints('config-feedforward.txt')


 ****** Running generation 0 ****** 



SystemExit: 